In [1]:
import pandas as pd
from pathlib import Path

bronze = Path("../data/bronze/transactions")
parts = sorted(bronze.glob("batch_date=*"))
len(parts), parts[0].name


(731, 'batch_date=2019-01-01')

In [2]:
day = pd.read_parquet(parts[0])
day.shape

(2414, 27)

In [3]:
day.dtypes


Unnamed: 0                                str
trans_date_trans_time                     str
cc_num                                    str
merchant                                  str
category                                  str
amt                                       str
first                                     str
last                                      str
gender                                    str
street                                    str
city                                      str
state                                     str
zip                                       str
lat                                       str
long                                      str
city_pop                                  str
job                                       str
dob                                       str
trans_num                                 str
unix_time                                 str
merch_lat                                 str
merch_long                        

In [4]:
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

from src.silver.transactions import load_pii_key, transform_transactions

key = load_pii_key()                      # never print this
silver_day = transform_transactions(day, key)
silver_day.shape

(2414, 22)

In [5]:
print(silver_day.dtypes)
print("hash lengths:", silver_day["card_hash"].str.len().unique())
print("zip lengths:", silver_day["zip"].str.len().unique())
print("fraud_ prefix left:", silver_day["merchant"].str.startswith("fraud_").sum())
print("cards  bronze/silver:", day["cc_num"].nunique(), silver_day["card_hash"].nunique())
print("fraud  bronze/silver:", (day["is_fraud"] == "1").sum(), silver_day["is_fraud"].sum())
print("birth years:", silver_day["birth_year"].min(), silver_day["birth_year"].max())
print("deterministic:", transform_transactions(day, key)["card_hash"].equals(silver_day["card_hash"]))

trans_num                        str
trans_ts              datetime64[us]
card_hash                        str
merchant                         str
category                         str
amt                          float64
gender                           str
city                             str
state                            str
zip                              str
lat                          float64
long                         float64
city_pop                       int64
job                              str
birth_year                     int16
merch_lat                    float64
merch_long                   float64
is_fraud                        int8
_ingested_at     datetime64[us, UTC]
_source                          str
_source_file                     str
_batch_date     date32[day][pyarrow]
dtype: object
hash lengths: [64]
zip lengths: [5]
fraud_ prefix left: 0
cards  bronze/silver: 779 779
fraud  bronze/silver: 0 0
birth years: 1926 2005
deterministic: True


In [6]:
leap = [p for p in parts if p.name == "batch_date=2020-02-29"][0]
empty_bronze = pd.read_parquet(leap)
empty_silver = transform_transactions(empty_bronze, key)
print(empty_bronze.shape, empty_silver.shape)
print((empty_silver.dtypes == silver_day.dtypes).all())

(0, 27) (0, 22)
True
